# Stage 3 (v2): Document Logic Analysis with FRIDA

This notebook extends `compute_metrics_v1.ipynb`.

New signals in v2:
- `semantic_drift_score` (global trajectory instability),
- `segment_centrality` and `outlierness`,
- `coverage_score` (how intro points are covered by main/conclusion),
- `novelty_redundancy_profile` (repeat vs jump behavior over sequence).


## 1) Imports and configuration

Define paths, thresholds, and runtime params.


In [ ]:
import json
from pathlib import Path

import numpy as np


BASE = Path('out')
META_PATH = BASE / 'meta.json'
EMB_PATH = BASE / 'emb_FRIDA.npy'
REPORT_PATH = BASE / 'document_metrics_report_v2.json'

TOP_N_TRANSITIONS = 5
TOP_N_OUTLIERS = 5
REDUNDANCY_THRESHOLD = 0.90
ADJ_OUTLIER_Z = -1.0

print('meta:', META_PATH)
print('emb :', EMB_PATH)
print('report:', REPORT_PATH)


## 2) Load data and build similarity matrix

Load metadata and embeddings, then compute cosine matrix `S` and adjacent similarities `adj`.


In [ ]:
with open(META_PATH, 'r', encoding='utf-8') as f:
    meta = json.load(f)

emb = np.load(EMB_PATH).astype(np.float32)

n_segments = len(meta)
if emb.shape[0] != n_segments:
    raise ValueError(f'Mismatch: emb={emb.shape[0]} meta={n_segments}')


def cosine_matrix(vectors: np.ndarray) -> np.ndarray:
    vectors = vectors / (np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-9)
    return vectors @ vectors.T


S = cosine_matrix(emb)
adj = np.array([float(S[i, i + 1]) for i in range(n_segments - 1)], dtype=np.float32)

roles = np.array([m.get('role', '') for m in meta])
paths = [str(m.get('path', '')) for m in meta]
parents = np.array([str(m.get('parent', '')) for m in meta])

main_idx = np.where(roles == 'main')[0]
intro_idx = np.where(roles == 'intro')[0]
conc_idx = np.where(roles == 'conclusion')[0]

intro_i = int(intro_idx[0]) if intro_idx.size else None
conc_i = int(conc_idx[0]) if conc_idx.size else None

print('segments:', n_segments)
print('S shape :', S.shape)
print('adj size:', adj.size)
print('roles   : main=', len(main_idx), 'intro=', intro_i, 'conc=', conc_i)


## 3) Baseline metrics from v1

Keep baseline signals for continuity:
- local adjacency stats,
- intro/main and conclusion/main consistency,
- redundancy in main body.


In [ ]:
def upper_triangle_values(M: np.ndarray) -> np.ndarray:
    if M.shape[0] < 2:
        return np.array([], dtype=np.float32)
    iu = np.triu_indices_from(M, k=1)
    return M[iu].astype(np.float32)


adj_mean = float(adj.mean()) if adj.size else float('nan')
adj_std = float(adj.std(ddof=0)) if adj.size else float('nan')
adj_z = (adj - adj_mean) / adj_std if np.isfinite(adj_std) and adj_std > 0 else np.zeros_like(adj)

weak_transition_indices = [int(i) for i, z in enumerate(adj_z) if float(z) <= ADJ_OUTLIER_Z]
weakest_order = np.argsort(adj)[: min(TOP_N_TRANSITIONS, adj.size)] if adj.size else np.array([], dtype=np.int64)

weakest_transitions = []
for i in weakest_order:
    i = int(i)
    weakest_transitions.append(
        {
            'from_index': i,
            'to_index': i + 1,
            'from_path': paths[i],
            'to_path': paths[i + 1],
            'adj_similarity': float(adj[i]),
            'adj_z': float(adj_z[i]),
        }
    )

main_main_mean = float('nan')
intro_to_main_mean = float('nan')
conc_to_main_mean = float('nan')
intro_delta = float('nan')
conc_delta = float('nan')

if main_idx.size:
    mm_vals = upper_triangle_values(S[np.ix_(main_idx, main_idx)])
    if mm_vals.size:
        main_main_mean = float(mm_vals.mean())

if intro_i is not None and main_idx.size:
    intro_to_main_mean = float(S[intro_i, main_idx].mean())

if conc_i is not None and main_idx.size:
    conc_to_main_mean = float(S[conc_i, main_idx].mean())

if np.isfinite(intro_to_main_mean) and np.isfinite(main_main_mean):
    intro_delta = float(intro_to_main_mean - main_main_mean)
if np.isfinite(conc_to_main_mean) and np.isfinite(main_main_mean):
    conc_delta = float(conc_to_main_mean - main_main_mean)

redund_90 = float('nan')
if main_idx.size >= 2:
    mm_vals = upper_triangle_values(S[np.ix_(main_idx, main_idx)])
    if mm_vals.size:
        redund_90 = float(np.mean(mm_vals > REDUNDANCY_THRESHOLD))

print('adj_mean:', round(adj_mean, 4))
print('intro_delta:', round(intro_delta, 4) if np.isfinite(intro_delta) else 'nan')
print('conc_delta :', round(conc_delta, 4) if np.isfinite(conc_delta) else 'nan')
print('redund_90  :', round(redund_90, 4) if np.isfinite(redund_90) else 'nan')


## 4) Semantic drift and trajectory smoothness

Go beyond neighbor similarity:
- `semantic_drift_score` from step sizes and directional turns,
- `smoothness_score` from second differences and `S[i, i+2]`.


In [ ]:
step_vecs = emb[1:] - emb[:-1]
step_norms = np.linalg.norm(step_vecs, axis=1) if step_vecs.size else np.array([], dtype=np.float32)

turn_sharpness = []
for i in range(len(step_vecs) - 1):
    a = step_vecs[i]
    b = step_vecs[i + 1]
    denom = (np.linalg.norm(a) * np.linalg.norm(b)) + 1e-9
    cos_ab = float(np.dot(a, b) / denom)
    turn_sharpness.append(1.0 - cos_ab)
turn_sharpness = np.array(turn_sharpness, dtype=np.float32)

step_mean = float(step_norms.mean()) if step_norms.size else 0.0
turn_mean = float(turn_sharpness.mean()) if turn_sharpness.size else 0.0
semantic_drift_score = float(0.6 * step_mean + 0.4 * turn_mean)

second_diff_norms = np.linalg.norm(emb[2:] - 2 * emb[1:-1] + emb[:-2], axis=1) if n_segments >= 3 else np.array([], dtype=np.float32)
second_diff_mean = float(second_diff_norms.mean()) if second_diff_norms.size else 0.0

skip2 = np.array([float(S[i, i + 2]) for i in range(n_segments - 2)], dtype=np.float32) if n_segments >= 3 else np.array([], dtype=np.float32)
skip2_mean = float(skip2.mean()) if skip2.size else float('nan')

smoothness_score = float((1.0 / (1.0 + second_diff_mean)) * 0.5 + (skip2_mean if np.isfinite(skip2_mean) else 0.0) * 0.5)

print('semantic_drift_score:', round(semantic_drift_score, 4))
print('smoothness_score    :', round(smoothness_score, 4))


## 5) Segment centrality and outliers

Find segments that are globally weakly connected.


In [ ]:
centrality_all = S.mean(axis=1).astype(np.float32)

if main_idx.size:
    centrality_main = S[:, main_idx].mean(axis=1).astype(np.float32)
else:
    centrality_main = np.full(n_segments, np.nan, dtype=np.float32)

outlierness = (centrality_all.mean() - centrality_all).astype(np.float32)

outlier_order = np.argsort(outlierness)[::-1][: min(TOP_N_OUTLIERS, n_segments)]
segment_outliers = []
for idx in outlier_order:
    idx = int(idx)
    segment_outliers.append(
        {
            'index': idx,
            'path': paths[idx],
            'role': str(roles[idx]),
            'centrality_all': float(centrality_all[idx]),
            'centrality_main': float(centrality_main[idx]) if np.isfinite(centrality_main[idx]) else None,
            'outlierness': float(outlierness[idx]),
        }
    )

print('centrality_all mean:', round(float(centrality_all.mean()), 4))
print('top outliers:', len(segment_outliers))


## 6) Coverage score

Measure whether intro points are covered by main/conclusion semantically.


In [ ]:
coverage_threshold = 0.70
if main_idx.size >= 2:
    mm_vals = upper_triangle_values(S[np.ix_(main_idx, main_idx)])
    if mm_vals.size:
        coverage_threshold = float(np.median(mm_vals))

intro_to_main_best = []
intro_to_conc_best = []

for i in intro_idx:
    i = int(i)
    if main_idx.size:
        intro_to_main_best.append(float(np.max(S[i, main_idx])))
    if conc_idx.size:
        intro_to_conc_best.append(float(np.max(S[i, conc_idx])))

intro_to_main_best = np.array(intro_to_main_best, dtype=np.float32)
intro_to_conc_best = np.array(intro_to_conc_best, dtype=np.float32)

coverage_main = float(np.mean(intro_to_main_best >= coverage_threshold)) if intro_to_main_best.size else float('nan')
coverage_conclusion = float(np.mean(intro_to_conc_best >= coverage_threshold)) if intro_to_conc_best.size else float('nan')

coverage_parts = [x for x in [coverage_main, coverage_conclusion] if np.isfinite(x)]
coverage_score = float(np.mean(coverage_parts)) if coverage_parts else float('nan')

print('coverage_threshold :', round(coverage_threshold, 4))
print('coverage_main      :', round(coverage_main, 4) if np.isfinite(coverage_main) else 'nan')
print('coverage_conclusion:', round(coverage_conclusion, 4) if np.isfinite(coverage_conclusion) else 'nan')
print('coverage_score     :', round(coverage_score, 4) if np.isfinite(coverage_score) else 'nan')


## 7) Novelty / redundancy profile over sequence

For each new segment, compare max similarity to previous content.


In [ ]:
novelty_profile = []
max_prev_sims = []

for i in range(1, n_segments):
    max_sim_prev = float(np.max(S[i, :i]))
    max_prev_sims.append(max_sim_prev)

max_prev_sims = np.array(max_prev_sims, dtype=np.float32)

if max_prev_sims.size:
    high_thr = float(np.quantile(max_prev_sims, 0.85))
    low_thr = float(np.quantile(max_prev_sims, 0.15))
else:
    high_thr = 0.90
    low_thr = 0.55

for i in range(1, n_segments):
    sim_prev = float(np.max(S[i, :i]))
    if sim_prev >= high_thr:
        label = 'redundant'
    elif sim_prev <= low_thr:
        label = 'jump'
    else:
        label = 'balanced'

    novelty_profile.append(
        {
            'index': i,
            'path': paths[i],
            'max_similarity_to_previous': sim_prev,
            'label': label,
        }
    )

redundant_share = float(np.mean([p['label'] == 'redundant' for p in novelty_profile])) if novelty_profile else float('nan')
jump_share = float(np.mean([p['label'] == 'jump' for p in novelty_profile])) if novelty_profile else float('nan')

print('novelty low/high thresholds:', round(low_thr, 4), '/', round(high_thr, 4))
print('redundant_share:', round(redundant_share, 4) if np.isfinite(redundant_share) else 'nan')
print('jump_share     :', round(jump_share, 4) if np.isfinite(jump_share) else 'nan')


## 8) Problem flags and risk score v2

Combine baseline and new signals into interpretable flags and a composite risk score.


In [ ]:
problems = []

if np.isfinite(adj_mean) and adj_mean < 0.65:
    problems.append({'type': 'low_local_coherence', 'severity': 'high', 'metric': 'adj_mean', 'value': float(adj_mean), 'threshold': 0.65})

if adj.size and (len(weak_transition_indices) / adj.size) > 0.20:
    problems.append({'type': 'many_weak_transitions', 'severity': 'high', 'metric': 'weak_transition_share', 'value': float(len(weak_transition_indices) / adj.size), 'threshold': 0.20})

if np.isfinite(intro_delta) and intro_delta < -0.05:
    problems.append({'type': 'intro_main_misalignment', 'severity': 'medium', 'metric': 'intro_delta', 'value': float(intro_delta), 'threshold': -0.05})

if np.isfinite(conc_delta) and conc_delta < -0.05:
    problems.append({'type': 'conclusion_main_misalignment', 'severity': 'medium', 'metric': 'conclusion_delta', 'value': float(conc_delta), 'threshold': -0.05})

if np.isfinite(redund_90) and redund_90 > 0.25:
    problems.append({'type': 'high_redundancy', 'severity': 'medium', 'metric': 'redund_90', 'value': float(redund_90), 'threshold': 0.25})

if np.isfinite(coverage_score) and coverage_score < 0.50:
    problems.append({'type': 'low_goal_coverage', 'severity': 'high', 'metric': 'coverage_score', 'value': float(coverage_score), 'threshold': 0.50})

if np.isfinite(jump_share) and jump_share > 0.30:
    problems.append({'type': 'high_semantic_jumps', 'severity': 'medium', 'metric': 'jump_share', 'value': float(jump_share), 'threshold': 0.30})

risk_score = 0.0

if np.isfinite(adj_mean):
    risk_score += max(0.0, (0.70 - adj_mean)) * 100.0

if adj.size:
    risk_score += (len(weak_transition_indices) / adj.size) * 30.0

if np.isfinite(redund_90):
    risk_score += max(0.0, redund_90 - 0.10) * 60.0

if np.isfinite(semantic_drift_score):
    risk_score += min(20.0, semantic_drift_score * 15.0)

if np.isfinite(smoothness_score):
    risk_score += max(0.0, (0.60 - smoothness_score)) * 35.0

if np.isfinite(coverage_score):
    risk_score += max(0.0, (0.70 - coverage_score)) * 45.0

if np.isfinite(jump_share):
    risk_score += jump_share * 30.0

risk_score = float(min(100.0, max(0.0, risk_score)))

print('problem flags:', len(problems))
print('risk_score_v2:', round(risk_score, 2))


## 9) Build and save report v2

Save expanded report with all metrics and profiles.


In [ ]:
report = {
    'version': 'v2',
    'model': 'FRIDA',
    'n_segments': int(n_segments),
    'local_coherence': {
        'adj_mean': float(adj_mean),
        'adj_std': float(adj_std),
        'weak_transition_count': int(len(weak_transition_indices)),
        'weak_transition_share': float(len(weak_transition_indices) / adj.size) if adj.size else None,
    },
    'structure_consistency': {
        'intro_to_main_mean': float(intro_to_main_mean) if np.isfinite(intro_to_main_mean) else None,
        'conclusion_to_main_mean': float(conc_to_main_mean) if np.isfinite(conc_to_main_mean) else None,
        'main_main_mean': float(main_main_mean) if np.isfinite(main_main_mean) else None,
        'intro_delta': float(intro_delta) if np.isfinite(intro_delta) else None,
        'conclusion_delta': float(conc_delta) if np.isfinite(conc_delta) else None,
    },
    'drift_and_smoothness': {
        'semantic_drift_score': float(semantic_drift_score),
        'step_mean_norm': float(step_mean),
        'turn_mean_sharpness': float(turn_mean),
        'second_diff_mean': float(second_diff_mean),
        'skip2_mean': float(skip2_mean) if np.isfinite(skip2_mean) else None,
        'smoothness_score': float(smoothness_score),
    },
    'centrality': {
        'centrality_all_mean': float(centrality_all.mean()),
        'top_outliers': segment_outliers,
    },
    'coverage': {
        'threshold': float(coverage_threshold),
        'coverage_main': float(coverage_main) if np.isfinite(coverage_main) else None,
        'coverage_conclusion': float(coverage_conclusion) if np.isfinite(coverage_conclusion) else None,
        'coverage_score': float(coverage_score) if np.isfinite(coverage_score) else None,
    },
    'novelty_redundancy_profile': {
        'high_threshold': float(high_thr),
        'low_threshold': float(low_thr),
        'redundant_share': float(redundant_share) if np.isfinite(redundant_share) else None,
        'jump_share': float(jump_share) if np.isfinite(jump_share) else None,
        'items': novelty_profile,
    },
    'redundancy': {
        'redund_90': float(redund_90) if np.isfinite(redund_90) else None,
        'threshold': float(REDUNDANCY_THRESHOLD),
    },
    'weakest_transitions_top_n': weakest_transitions,
    'problem_flags': problems,
    'risk_score_v2': float(risk_score),
}

REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')

print('saved:', REPORT_PATH)


## 10) Quick summary

Compact printout for manual inspection.


In [ ]:
print('MODEL:', report['model'])
print('VERSION:', report['version'])
print('SEGMENTS:', report['n_segments'])
print('RISK SCORE V2:', round(report['risk_score_v2'], 2))

print('')
print('Key metrics:')
print('- adj_mean:', round(report['local_coherence']['adj_mean'], 4))
print('- semantic_drift_score:', round(report['drift_and_smoothness']['semantic_drift_score'], 4))
print('- smoothness_score:', round(report['drift_and_smoothness']['smoothness_score'], 4))
print('- coverage_score:', report['coverage']['coverage_score'])
print('- redundant_share:', report['novelty_redundancy_profile']['redundant_share'])
print('- jump_share:', report['novelty_redundancy_profile']['jump_share'])

print('')
print('Problem flags:')
if report['problem_flags']:
    for p in report['problem_flags']:
        print('-', p['type'], '|', p['metric'], '=', round(float(p['value']), 4))
else:
    print('- none')
